# Sila na četvrtcilindričnu plohu — predvidi, izračunaj, provjeri

**Poglavlje U05: hidrostatske sile na zakrivljene plohe**

Četvrtkružni luk širine $L$ parametriziramo kutom $0\leq\theta\leq\pi/2$.
Voda kvasi konveksnu donju i lijevu stranu, pa normala **iz vode prema plohi**
ima komponente $(\cos\theta,\sin\theta)$: $+x$ je desno, $+z$ prema gore.


## 1. Predvidi

1. Koji predznak očekuješ za $F_x$ i $F_z$ uz navedenu okupanu stranu?
2. Ako se voda premjesti na suprotnu stranu iste plohe, mijenja li se samo
   iznos ili i smjer sile?
3. Zašto rezultanta kružnog luka mora prolaziti središtem zakrivljenosti?

Najprije nacrtaj barem dvije lokalne tlačne strelice.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
RHO, G = 998.0, 9.81

def analiticke_komponente(R, h_t, L, strana=1.0):
    Fx = strana*RHO*G*L*R*(h_t + R/2)
    Fz = strana*RHO*G*L*R*(h_t + np.pi*R/4)
    return Fx, Fz

R, h_t, L = 0.80, 1.00, 2.00
Fx_ref, Fz_ref = analiticke_komponente(R, h_t, L)
print(f"Analitički: Fx = {Fx_ref/1000:.3f} kN, Fz = {Fz_ref/1000:.3f} kN")
print(f"smjer rezultante = {np.degrees(np.arctan2(Fz_ref, Fx_ref)):.2f}°")


## 2. Izračunaj — vektorska integracija po luku

Dubina točke luka je $h(\theta)=h_t+R\sin\theta$, tlak je $p=\rho gh$, a
element površine $dA=LR\,d\theta$. Zato numerički zbrajamo

$$dF_x=p\cos\theta\,dA,\qquad dF_z=p\sin\theta\,dA.$$

Trapezno pravilo mora konvergirati referentnim komponentama, ali predznak
dolazi isključivo iz dogovorene normale i okupane strane.


In [ ]:
def trapz_local(y, x):
    return np.sum(0.5*(y[:-1]+y[1:]) * np.diff(x))

def integriraj_luk(R, h_t, L, n, strana=1.0):
    theta = np.linspace(0.0, np.pi/2, n+1)
    p = RHO*G*(h_t + R*np.sin(theta))
    dF_skala = p*L*R
    Fx = trapz_local(strana*dF_skala*np.cos(theta), theta)
    Fz = trapz_local(strana*dF_skala*np.sin(theta), theta)
    # Središte je na (R, -h_t); r_rel je suprotan od odabrane normale.
    rx, rz = -R*np.cos(theta), -R*np.sin(theta)
    gusto_momenta = rx*(strana*dF_skala*np.sin(theta)) \
        - rz*(strana*dF_skala*np.cos(theta))
    M_C = trapz_local(gusto_momenta, theta)
    return Fx, Fz, M_C

n_mreza = np.array([8, 16, 32, 64, 128, 256])
num = np.array([integriraj_luk(R, h_t, L, int(n)) for n in n_mreza])
err = np.sqrt((num[:,0]-Fx_ref)**2 + (num[:,1]-Fz_ref)**2)
red = np.log2(err[:-1]/err[1:])
print("n      Fx [kN]    Fz [kN]    pogreška rezultante [N]")
for n, (fx, fz, _), e in zip(n_mreza, num, err):
    print(f"{n:3d}  {fx/1000:10.5f} {fz/1000:10.5f} {e:20.6e}")

fig, ax = plt.subplots(figsize=(6.8, 3.8))
ax.loglog((np.pi*R/2)/n_mreza, err, "o-")
ax.set(xlabel="duljina elementa luka (m)", ylabel="vektorska pogreška sile (N)",
       title="Konvergencija integracije po zakrivljenoj plohi")
ax.grid(ls=":", which="both", alpha=0.6)
plt.show()


## 3. Provjeri — predznaci, projekcija i moment

Horizontalnu magnitudu neovisno daje sila na vertikalnu projekciju. Moment
rezultante o središtu kružnog luka mora biti nula jer svaka lokalna sila
prolazi tim središtem. Promjena okupane strane mora obrnuti oba predznaka.


In [ ]:
Fx_num, Fz_num, M_C = integriraj_luk(R, h_t, L, 2048, strana=1.0)
Fx_proj = RHO*G*(L*R)*(h_t+R/2)
Fx_sup, Fz_sup, _ = integriraj_luk(R, h_t, L, 2048, strana=-1.0)

print(f"Fx iz projekcije = {Fx_proj/1000:.6f} kN")
print(f"moment o središtu C = {M_C:.3e} N m")
print(f"opaženi red zadnja tri refiniranja = {red[-3:]}")

assert np.isclose(Fx_num, Fx_proj, rtol=2e-7)
assert abs(M_C) < 1e-8
assert Fx_num > 0 and Fz_num > 0
assert np.allclose([Fx_sup, Fz_sup], [-Fx_num, -Fz_num], rtol=1e-12)
assert np.all((red[-3:] > 1.99) & (red[-3:] < 2.01))
print("PASS: projekcija, moment, predznaci i drugi red konvergencije.")


## Granica modela

Formula pomoćnog volumena daje magnitudu vertikalne komponente, ne njezin
smjer. Smjer uvijek odredi iz stvarne okupane strane. Rezultanta prolazi
središtem samo za kružni luk; to nije svojstvo proizvoljne zakrivljene plohe.
